# Example 8.1.2 / 2: Find Each Solver's Options

Companion notebook for **8.1.2 Nonlinear and Global Solvers** in the AMPL lab Documentation, section "Find Each Solver's Options". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Find Each Solver's Options

Every solver reads its options from the AMPL option `<solver>_options`, or from the matching
`ampl.solve(..., <solver>_options="...")` argument, as a space-separated list of `name=value`
pairs. The class each solver is documented for, and one option from its options page:

| Solver | Documented for | Module | Options page | Example setting and documented effect |
|---|---|---|---|---|
| BARON | nonconvex NLP and MINLP to global optimality | `baron` | [BARON options](https://dev.ampl.com/solvers/baron/options.html) | `baron_options 'maxtime=60'`: at most 60 CPU seconds (default 1000; -1 means no limit) |
| BARONMP | the experimental BARON driver with MP modeling features | `baron` | [BARONMP options](https://dev.ampl.com/solvers/baron/baronmp.html) | `baronmp_options 'lim:time=60'`: at most 60 seconds; `maxtime` is an alias |
| Knitro | local optimization of large smooth or discrete models | `knitro` | [Knitro options](https://dev.ampl.com/solvers/knitro/options.html) | `knitro_options 'maxtime=60'`: at most 60 seconds of real time |
| CONOPT | highly nonlinear continuous models | `conopt` | [CONOPT options](https://dev.ampl.com/solvers/conopt/options.html) | `conopt_options 'outlev=3'`: a log line every `logfreq` iterations |
| LGO | global search over continuous variables | `lgo` | [LGO options](https://dev.ampl.com/solvers/lgo/options.html) | `lgo_options 'timelim=60'`: at most 60 seconds (default 300) |
| Bonmin | MINLP; exact for convex models, a heuristic otherwise | `coin` | [Bonmin options](https://dev.ampl.com/solvers/bonmin/options.html) | `bonmin_options 'bonmin.algorithm=B-BB'`: nonlinear branch-and-bound; run below |
| Couenne | global optimum of nonconvex MINLP by spatial branch-and-bound | `coin` | [Couenne options](https://dev.ampl.com/solvers/couenne/options.html) | `couenne_options 'bonmin.time_limit=60'`: at most 60 seconds; run below |
| Ipopt | large-scale continuous NLP by an interior-point method | `coin` | [Ipopt options](https://dev.ampl.com/solvers/ipopt/options.html) | `ipopt_options 'max_iter=2'`: stop after 2 iterations; run below |

BARON, Knitro, CONOPT and LGO need a commercial or trial license, so their rows give the
documented effect and were not run here. The three `coin` solvers run with the Community Edition.
This example adds an integer variable `k` to the model above, so Bonmin and Couenne have an
MINLP to solve, and starts every run from $x = 1$:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
var x >= -2, <= 2;                         # continuous decision
var k integer >= -2, <= 2;                 # integer decision
minimize g: x^4 - 3 * x^2 + x + (x - k)^2; # nonconvex MINLP objective

Writing model.mod


### amplpy: run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# run.py
runs = [("ipopt", "max_iter=2"), ("bonmin", "bonmin.algorithm=B-BB"),
        ("couenne", "bonmin.time_limit=60")]
for solver, options in runs:
    ampl.var["x"].set_value(1)
    ampl.solve(solver=solver, verbose=False, **{solver + "_options": options})
    print(f"{solver} {options}: {ampl.solve_result}, g = {ampl.obj['g'].value():.4f}")

ipopt max_iter=2: limit, g = -1.0636
bonmin bonmin.algorithm=B-BB: solved, g = -1.0562
couenne bonmin.time_limit=60: solved, g = -3.4348
